<a href="https://colab.research.google.com/github/syedadilejazz/Complete_GenAI_Series_Bappy_euron/blob/main/Text_Summarization_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri Jul 10 12:00:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 15.2 MB/s eta 0:00:00


In [6]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate
!pip install evaluate

Found existing installation: transformers 5.13.0
Uninstalling transformers-5.13.0:
  Successfully uninstalled transformers-5.13.0
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
  Using cached transformers-5.13.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.13.0-py3-none-any.whl (11.5 MB)
Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [2]:
from transformers import pipeline,set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [3]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
model_ckpt="google/pegasus-cnn_dailymail"

tokenizer=AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [5]:
model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)#device=GPU

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [7]:
dataset=load_dataset("koushik7198/Samsung-samsum_processed")

README.md:   0%|          | 0.00/560 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.52M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9032 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3227 [00:00<?, ? examples/s]

In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 9032
    })
    validation: Dataset({
        features: ['input', 'output'],
        num_rows: 3872
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 3227
    })
})

In [13]:
dataset["train"][0]["input"]

"Aiden: hi!\r\nAiden: wanna hang out?\r\nAmelia: I've told you a milion times - I'm not interested. I have a boyfriend.\r\nAmelia: Stop being so pushy, it's not gonna work.\r\nAiden: stop being so beautiful. :*\r\nAmelia: Jesus, just get lost!!"

In [14]:
dataset["train"][0]["output"]

"Amelia doesn't want to hang out with Aiden. She has a boyfriend. "

In [20]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['input'], max_length=1024, truncation=True )

    target_encodings = tokenizer(example_batch['output'], max_length=128, truncation=True )

    return {
        'input_ids' : input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }

In [21]:
dataset_samsum_pt=dataset.map(convert_examples_to_features,batched=True)

Map:   0%|          | 0/9032 [00:00<?, ? examples/s]

Map:   0%|          | 0/3872 [00:00<?, ? examples/s]

Map:   0%|          | 0/3227 [00:00<?, ? examples/s]

In [23]:
dataset_samsum_pt["train"]

Dataset({
    features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 9032
})

In [25]:
dataset_samsum_pt["train"]['input_ids'][1]

[1185,
 151,
 325,
 109,
 829,
 152,
 3292,
 151,
 398,
 243,
 108,
 14484,
 147,
 1185,
 151,
 8132,
 213,
 154,
 8537,
 3292,
 151,
 2604,
 4706,
 131,
 144,
 4911,
 186,
 152,
 1185,
 151,
 33894,
 43114,
 3292,
 151,
 520,
 3167,
 112,
 1232,
 120,
 107,
 1894,
 108,
 126,
 547,
 122,
 181,
 523,
 3818,
 107,
 1185,
 151,
 12098,
 108,
 268,
 107,
 475,
 111,
 523,
 3818,
 107,
 3292,
 151,
 6610,
 147,
 184,
 19246,
 111,
 19246,
 430,
 145,
 419,
 1909,
 252,
 147,
 1185,
 151,
 1439,
 172,
 126,
 147,
 3292,
 151,
 14557,
 11472,
 1159,
 126,
 131,
 116,
 114,
 234,
 641,
 112,
 3540,
 299,
 109,
 2243,
 190,
 109,
 1536,
 107,
 1185,
 151,
 33894,
 147,
 3292,
 151,
 125,
 235,
 147,
 343,
 178,
 368,
 126,
 147,
 325,
 688,
 140,
 172,
 233,
 120,
 131,
 116,
 546,
 147,
 1185,
 151,
 1414,
 131,
 144,
 823,
 213,
 647,
 997,
 368,
 126,
 152,
 3292,
 151,
 1894,
 108,
 181,
 2266,
 7353,
 126,
 108,
 155,
 156,
 3175,
 112,
 109,
 1030,
 111,
 280,
 589,
 266,
 126,
 107,
 11

In [29]:
#Training
from transformers import DataCollatorForSeq2Seq#Load data in batches

seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)